In [2]:
import polars as pl

lf = pl.scan_parquet(
    "out/sctv-out-snapshot-20261102-074800.parquet"
)

print(
    lf.select(
        pl.len().alias("total_events"),
        pl.col("channel_id").n_unique().alias("channels"),
        pl.col("is_future").sum().alias("future_events"),
        pl.col("hot").sum().alias("hot_events"),
        pl.col("name").null_count().alias("missing_names"),
    ).collect()
)

shape: (1, 5)
┌──────────────┬──────────┬───────────────┬────────────┬───────────────┐
│ total_events ┆ channels ┆ future_events ┆ hot_events ┆ missing_names │
│ ---          ┆ ---      ┆ ---           ┆ ---        ┆ ---           │
│ u32          ┆ u32      ┆ u32           ┆ u32        ┆ u32           │
╞══════════════╪══════════╪═══════════════╪════════════╪═══════════════╡
│ 4242292      ┆ 39       ┆ 23347         ┆ 0          ┆ 0             │
└──────────────┴──────────┴───────────────┴────────────┴───────────────┘


In [3]:
lf.collect_schema()

Schema([('filename_source', String),
        ('is_future', Boolean),
        ('channel_id', Int32),
        ('date_aspnet_str', String),
        ('start_time', String),
        ('end_time', String),
        ('name', String),
        ('short_descriptor', String),
        ('extended_descriptor', String),
        ('hot', Boolean)])

In [4]:
result = lf.select(
    pl.col("short_descriptor").null_count().alias("null_short"),
    pl.col("extended_descriptor").null_count().alias("null_extended"),
    pl.col("hot").null_count().alias("null_hot"),
    (
        pl.col("name") == pl.col("short_descriptor")
    ).sum().alias("name_eq_short"),
    (
        pl.col("name") == pl.col("extended_descriptor")
    ).sum().alias("name_eq_extended"),
).collect()

print(result)

shape: (1, 5)
┌────────────┬───────────────┬──────────┬───────────────┬──────────────────┐
│ null_short ┆ null_extended ┆ null_hot ┆ name_eq_short ┆ name_eq_extended │
│ ---        ┆ ---           ┆ ---      ┆ ---           ┆ ---              │
│ u32        ┆ u32           ┆ u32      ┆ u32           ┆ u32              │
╞════════════╪═══════════════╪══════════╪═══════════════╪══════════════════╡
│ 0          ┆ 0             ┆ 0        ┆ 4242292       ┆ 4242292          │
└────────────┴───────────────┴──────────┴───────────────┴──────────────────┘


In [5]:
lf.head(100).collect()

filename_source,is_future,channel_id,date_aspnet_str,start_time,end_time,name,short_descriptor,extended_descriptor,hot
str,bool,i32,str,str,str,str,str,str,bool
"""data/96/2021-05/25.json""",false,96,"""/Date(1621875600000)/""","""00:20:00""","""01:10:00""","""CULINARY ASIA : ẨM THỰC CHÂU Á""","""CULINARY ASIA : ẨM THỰC CHÂU Á""","""CULINARY ASIA : ẨM THỰC CHÂU Á""",false
"""data/96/2021-05/25.json""",false,96,"""/Date(1621875600000)/""","""01:10:00""","""02:00:00""","""CRIME SCIENCE : KHOA HỌC TỘI P…","""CRIME SCIENCE : KHOA HỌC TỘI P…","""CRIME SCIENCE : KHOA HỌC TỘI P…",false
"""data/96/2021-05/25.json""",false,96,"""/Date(1621875600000)/""","""02:00:00""","""02:50:00""","""WOK THE WORLD - RICE UNEARTHED…","""WOK THE WORLD - RICE UNEARTHED…","""WOK THE WORLD - RICE UNEARTHED…",false
"""data/96/2021-05/25.json""",false,96,"""/Date(1621875600000)/""","""02:50:00""","""03:40:00""","""CULINARY ASIA : ẨM THỰC CHÂU Á""","""CULINARY ASIA : ẨM THỰC CHÂU Á""","""CULINARY ASIA : ẨM THỰC CHÂU Á""",false
"""data/96/2021-05/25.json""",false,96,"""/Date(1621875600000)/""","""03:40:00""","""04:30:00""","""CRIME SCIENCE : KHOA HỌC TỘI P…","""CRIME SCIENCE : KHOA HỌC TỘI P…","""CRIME SCIENCE : KHOA HỌC TỘI P…",false
…,…,…,…,…,…,…,…,…,…
"""data/96/2021-05/18.json""",false,96,"""/Date(1621270800000)/""","""03:40:00""","""04:05:00""","""CHINA'S SCIENCE REVOLUTION : C…","""CHINA'S SCIENCE REVOLUTION : C…","""CHINA'S SCIENCE REVOLUTION : C…",false
"""data/96/2021-05/18.json""",false,96,"""/Date(1621270800000)/""","""04:05:00""","""04:30:00""","""CHINA'S SCIENCE REVOLUTION : C…","""CHINA'S SCIENCE REVOLUTION : C…","""CHINA'S SCIENCE REVOLUTION : C…",false
"""data/96/2021-05/18.json""",false,96,"""/Date(1621270800000)/""","""04:30:00""","""05:30:00""","""ORANGUTAN JUNGLE SCHOOL (S2) :…","""ORANGUTAN JUNGLE SCHOOL (S2) :…","""ORANGUTAN JUNGLE SCHOOL (S2) :…",false


In [6]:
# Collecting channels
channels_df = pl.read_parquet("channels.parquet")
channels_df


Ma,Ten,TenNgan,Slogan,GioiThieu,NgayTao,LogoKenh,LichPhatSong,TrangThai,SEO_Title,SEO_Description,URL
i64,str,str,null,str,str,bool,bool,i64,null,null,str
94,"""Animal Planet""","""Animal Planet""",null,"""""","""/Date(1513887399077)/""",false,true,1,null,null,"""/animal-planet"""
50,"""AXN""","""AXN""",null,"""AXN là kênh phim hành động và …","""/Date(1523939652587)/""",false,true,1,null,null,"""/axn"""
90,"""Cartoon Network""","""Cartoon Network""",null,"""""","""/Date(1513887399077)/""",false,true,1,null,null,"""/cartoon-network"""
96,"""Discovery Asia""","""Discovery Asia""",null,"""""","""/Date(1513887399077)/""",false,true,1,null,null,"""/discovery-asia"""
187,"""Discovery Asia HD""","""DiscoveryAsiaHD""",null,"""""","""/Date(1513887399077)/""",false,true,1,null,null,""""""
…,…,…,…,…,…,…,…,…,…,…,…
182,"""SCTV6 HD - SNTV""","""SCTV6HD""",null,"""""","""/Date(1513887399077)/""",true,true,1,null,null,""""""
13,"""SCTV7 - Show""","""SCTV7""",null,"""SCTV7 là kênh Sân khấu, văn ng…","""/Date(1523939650887)/""",true,true,1,null,null,"""/sctv7"""
175,"""SCTV7 HD - Kênh sân khấu ""","""SCTV7HD""",null,"""""","""/Date(1513887399077)/""",true,true,1,null,null,""""""


In [7]:
validated = lf.with_columns(
    pl.col("filename_source")
      .str.extract(r"data/(\d+)/", 1)
      .cast(pl.Int32)
      .alias("path_channel_id"),

    pl.col("filename_source")
      .str.extract(r"data/\d+/(\d{4}-\d{2}/\d{2})\.json$", 1)
      .str.replace("/", "-")
      .str.to_date("%Y-%m-%d", strict=False)
      .alias("path_date"),
)

print(
    validated.select(
        pl.len().alias("events"),
        (
            pl.col("channel_id") != pl.col("path_channel_id")
        ).sum().alias("channel_mismatches"),
        pl.col("path_date").null_count().alias("invalid_paths"),
    ).collect()
)

shape: (1, 3)
┌─────────┬────────────────────┬───────────────┐
│ events  ┆ channel_mismatches ┆ invalid_paths │
│ ---     ┆ ---                ┆ ---           │
│ u32     ┆ u32                ┆ u32           │
╞═════════╪════════════════════╪═══════════════╡
│ 4242292 ┆ 0                  ┆ 0             │
└─────────┴────────────────────┴───────────────┘


In [8]:
# normalize start_time and end_time to datetime with timezone
normalized = validated.with_columns(
    pl.col("start_time")
      .str.to_time("%H:%M:%S", strict=False)
      .alias("start_t"),

    pl.col("end_time")
      .str.to_time("%H:%M:%S", strict=False)
      .alias("end_t"),
)

normalized = normalized.with_columns(
    pl.col("path_date")
      .dt.combine(pl.col("start_t"))
      .dt.replace_time_zone("Asia/Ho_Chi_Minh")
      .alias("start_datetime"),

    pl.col("path_date")
      .dt.combine(pl.col("end_t"))
      .dt.replace_time_zone("Asia/Ho_Chi_Minh")
      .alias("end_datetime"),
)

normalized = normalized.with_columns(
    pl.when(pl.col("end_datetime") < pl.col("start_datetime"))
      .then(pl.col("end_datetime").dt.offset_by("1d"))
      .otherwise(pl.col("end_datetime"))
      .alias("end_datetime")
)

normalized = normalized.with_columns(
    (
        pl.col("end_datetime") - pl.col("start_datetime")
    ).dt.total_seconds().alias("duration_secs")
)

In [13]:
normalized.head(5).collect()

filename_source,is_future,channel_id,date_aspnet_str,start_time,end_time,name,short_descriptor,extended_descriptor,hot,path_channel_id,path_date,start_t,end_t,start_datetime,end_datetime,duration_secs
str,bool,i32,str,str,str,str,str,str,bool,i32,date,time,time,"datetime[μs, Asia/Ho_Chi_Minh]","datetime[μs, Asia/Ho_Chi_Minh]",i64
"""data/96/2021-05/25.json""",false,96,"""/Date(1621875600000)/""","""00:20:00""","""01:10:00""","""CULINARY ASIA : ẨM THỰC CHÂU Á""","""CULINARY ASIA : ẨM THỰC CHÂU Á""","""CULINARY ASIA : ẨM THỰC CHÂU Á""",false,96,2021-05-25,00:20:00,01:10:00,2021-05-25 00:20:00 +07,2021-05-25 01:10:00 +07,3000
"""data/96/2021-05/25.json""",false,96,"""/Date(1621875600000)/""","""01:10:00""","""02:00:00""","""CRIME SCIENCE : KHOA HỌC TỘI P…","""CRIME SCIENCE : KHOA HỌC TỘI P…","""CRIME SCIENCE : KHOA HỌC TỘI P…",false,96,2021-05-25,01:10:00,02:00:00,2021-05-25 01:10:00 +07,2021-05-25 02:00:00 +07,3000
"""data/96/2021-05/25.json""",false,96,"""/Date(1621875600000)/""","""02:00:00""","""02:50:00""","""WOK THE WORLD - RICE UNEARTHED…","""WOK THE WORLD - RICE UNEARTHED…","""WOK THE WORLD - RICE UNEARTHED…",false,96,2021-05-25,02:00:00,02:50:00,2021-05-25 02:00:00 +07,2021-05-25 02:50:00 +07,3000
"""data/96/2021-05/25.json""",false,96,"""/Date(1621875600000)/""","""02:50:00""","""03:40:00""","""CULINARY ASIA : ẨM THỰC CHÂU Á""","""CULINARY ASIA : ẨM THỰC CHÂU Á""","""CULINARY ASIA : ẨM THỰC CHÂU Á""",false,96,2021-05-25,02:50:00,03:40:00,2021-05-25 02:50:00 +07,2021-05-25 03:40:00 +07,3000
"""data/96/2021-05/25.json""",false,96,"""/Date(1621875600000)/""","""03:40:00""","""04:30:00""","""CRIME SCIENCE : KHOA HỌC TỘI P…","""CRIME SCIENCE : KHOA HỌC TỘI P…","""CRIME SCIENCE : KHOA HỌC TỘI P…",false,96,2021-05-25,03:40:00,04:30:00,2021-05-25 03:40:00 +07,2021-05-25 04:30:00 +07,3000


In [14]:
print(
    normalized.select(
        pl.len().alias("total_events"),
        pl.col("path_date").null_count().alias("invalid_dates"),
        pl.col("start_t").null_count().alias("invalid_starts"),
        pl.col("end_t").null_count().alias("invalid_ends"),
        pl.col("duration_secs").min().alias("min_duration"),
        pl.col("duration_secs").max().alias("max_duration"),
        (pl.col("duration_secs") <= 0)
            .sum().alias("nonpositive_durations"),
    ).collect()
)

shape: (1, 7)
┌──────────────┬─────────────┬─────────────┬─────────────┬─────────────┬─────────────┬─────────────┐
│ total_events ┆ invalid_dat ┆ invalid_sta ┆ invalid_end ┆ min_duratio ┆ max_duratio ┆ nonpositive │
│ ---          ┆ es          ┆ rts         ┆ s           ┆ n           ┆ n           ┆ _durations  │
│ u32          ┆ ---         ┆ ---         ┆ ---         ┆ ---         ┆ ---         ┆ ---         │
│              ┆ u32         ┆ u32         ┆ u32         ┆ i64         ┆ i64         ┆ u32         │
╞══════════════╪═════════════╪═════════════╪═════════════╪═════════════╪═════════════╪═════════════╡
│ 4242292      ┆ 0           ┆ 0           ┆ 4           ┆ 0           ┆ 85500       ┆ 41661       │
└──────────────┴─────────────┴─────────────┴─────────────┴─────────────┴─────────────┴─────────────┘


In [15]:
print(
    normalized.filter(
        pl.col("duration_secs") <= 0
    )
    .group_by("start_time", "end_time")
    .agg(pl.len().alias("count"))
    .sort("count", descending=True)
    .limit(20)
    .collect()
)
print(
    normalized.filter(
        pl.col("end_t").is_null()
    )
    .select(
        "filename_source",
        "channel_id",
        "start_time",
        "end_time",
        "name",
    )
    .collect()
)
print(
    normalized.filter(
        pl.col("duration_secs") > 6 * 3600
    )
    .select(
        "filename_source",
        "channel_id",
        "start_time",
        "end_time",
        "duration_secs",
        "name",
    )
    .sort("duration_secs", descending=True)
    .limit(30)
    .collect()
)

shape: (20, 3)
┌────────────┬──────────┬───────┐
│ start_time ┆ end_time ┆ count │
│ ---        ┆ ---      ┆ ---   │
│ str        ┆ str      ┆ u32   │
╞════════════╪══════════╪═══════╡
│ 00:00:00   ┆ 00:00:00 ┆ 1743  │
│ 18:00:00   ┆ 18:00:00 ┆ 1149  │
│ 06:00:00   ┆ 06:00:00 ┆ 1115  │
│ 22:00:00   ┆ 22:00:00 ┆ 1113  │
│ 13:00:00   ┆ 13:00:00 ┆ 1071  │
│ …          ┆ …        ┆ …     │
│ 00:29:00   ┆ 00:29:00 ┆ 154   │
│ 00:28:00   ┆ 00:28:00 ┆ 146   │
│ 13:04:00   ┆ 13:04:00 ┆ 141   │
│ 13:08:00   ┆ 13:08:00 ┆ 130   │
│ 00:13:00   ┆ 00:13:00 ┆ 128   │
└────────────┴──────────┴───────┘
shape: (4, 5)
┌─────────────────────────┬────────────┬────────────┬──────────┬────────────────────┐
│ filename_source         ┆ channel_id ┆ start_time ┆ end_time ┆ name               │
│ ---                     ┆ ---        ┆ ---        ┆ ---      ┆ ---                │
│ str                     ┆ i32        ┆ str        ┆ str      ┆ str                │
╞═════════════════════════╪════════════╪═════════

In [17]:
import polars as pl

temporal = (
    lf.with_columns(
        pl.col("start_time")
        .str.to_time("%H:%M:%S", strict=False)
        .alias("start_t"),

        pl.when(pl.col("end_time") == "24:00:00")
        .then(pl.lit("00:00:00"))
        .otherwise(pl.col("end_time"))
        .str.to_time("%H:%M:%S", strict=False)
        .alias("end_t"),
    )
    .with_columns(
        pl.when(pl.col("end_time") == "24:00:00")
        .then(pl.lit("end_of_day"))
        .when(pl.col("start_t").is_null() | pl.col("end_t").is_null())
        .then(pl.lit("invalid"))
        .when(pl.col("end_t") == pl.col("start_t"))
        .then(pl.lit("zero"))
        .when(pl.col("end_t") > pl.col("start_t"))
        .then(pl.lit("normal"))
        .when(
            (pl.col("start_t") >= pl.time(20, 0))
            & (pl.col("end_t") <= pl.time(6, 0))
        )
        .then(pl.lit("potential_midnight"))
        .otherwise(pl.lit("suspicious_reversal"))
        .alias("time_class")
    )
)

print(
    temporal.group_by("time_class")
    .agg(pl.len().alias("events"))
    .sort("events", descending=True)
    .collect()
)
import polars as pl

analysis = temporal.with_columns(
    (
        (
            pl.col("end_t").cast(pl.Int64)
            - pl.col("start_t").cast(pl.Int64)
        ) / 1_000_000_000
    ).alias("raw_duration_secs")
).with_columns(
    pl.when(pl.col("time_class") == "end_of_day")
      .then(
          86400 - pl.col("start_t").cast(pl.Int64) / 1_000_000_000
      )
      .when(pl.col("raw_duration_secs") < 0)
      .then(pl.col("raw_duration_secs") + 86400)
      .otherwise(pl.col("raw_duration_secs"))
      .alias("adjusted_duration_secs")
)

print(
    analysis.group_by("time_class")
    .agg(
        pl.len().alias("events"),
        pl.col("adjusted_duration_secs").median().alias("median_secs"),
        pl.col("adjusted_duration_secs").max().alias("max_secs"),
        (pl.col("adjusted_duration_secs") > 6 * 3600)
            .sum().alias("over_6h"),
    )
    .sort("events", descending=True)
    .collect()
)

shape: (5, 2)
┌─────────────────────┬─────────┐
│ time_class          ┆ events  │
│ ---                 ┆ ---     │
│ str                 ┆ u32     │
╞═════════════════════╪═════════╡
│ normal              ┆ 4122577 │
│ potential_midnight  ┆ 77756   │
│ zero                ┆ 41661   │
│ suspicious_reversal ┆ 294     │
│ end_of_day          ┆ 4       │
└─────────────────────┴─────────┘
shape: (5, 5)
┌─────────────────────┬─────────┬─────────────┬──────────┬─────────┐
│ time_class          ┆ events  ┆ median_secs ┆ max_secs ┆ over_6h │
│ ---                 ┆ ---     ┆ ---         ┆ ---      ┆ ---     │
│ str                 ┆ u32     ┆ f64         ┆ f64      ┆ u32     │
╞═════════════════════╪═════════╪═════════════╪══════════╪═════════╡
│ normal              ┆ 4122577 ┆ 2700.0      ┆ 66600.0  ┆ 437     │
│ potential_midnight  ┆ 77756   ┆ 3000.0      ┆ 18900.0  ┆ 0       │
│ zero                ┆ 41661   ┆ 0.0         ┆ 0.0      ┆ 0       │
│ suspicious_reversal ┆ 294     ┆ 45000.0     